# NLA Inferencia — Batch
Extrae activaciones de la capa 20 de Qwen-2.5-7B para todos los prompts del JSON,
en ambos idiomas (ES y EN). Produce:
- `{id}_{lang}.npy` por cada prompt-idioma
- `metadatos_activaciones.json` con toda la info para el Verbalizer

In [ ]:
import torch, os, json
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ── Rutas ── ajustar si la carpeta HACKATHON no está en MyDrive raíz
HACKATHON     = '/content/drive/MyDrive/HACKATHON'
CHECKPOINT    = '/content/drive/MyDrive/nla_pipeline/checkpoints/qwen_sujeto'
PROMPTS_JSON  = f'{HACKATHON}/prompts_espanol_ingles.json'
DIR_SALIDA    = f'{HACKATHON}/activaciones'
CAPA          = 20   # capa media del modelo (28 capas en total)

os.makedirs(DIR_SALIDA, exist_ok=True)

vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
MODO_8BIT = vram_gb < 20

print(f'GPU        : {torch.cuda.get_device_name(0)}')
print(f'VRAM       : {vram_gb:.1f} GB')
print(f'Modo       : {"8-bit (T4)" if MODO_8BIT else "bfloat16 (Pro)"}')
print(f'Prompts    : {PROMPTS_JSON}')
print(f'Salida     : {DIR_SALIDA}')

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print('Cargando tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, trust_remote_code=True)

print(f'Cargando modelo en modo {"8-bit" if MODO_8BIT else "bfloat16"}...')
if MODO_8BIT:
    modelo = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT,
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map='auto',
        trust_remote_code=True,
    )
else:
    modelo = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT,
        torch_dtype=torch.bfloat16,
        device_map='cuda:0',
        trust_remote_code=True,
    )

modelo.eval()
print(f'✓ Modelo cargado — {modelo.config.num_hidden_layers} capas')
usado = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM usada : {usado:.1f} / {total:.1f} GB')

In [ ]:
def extraer_activacion(texto):
    """
    Corre un texto por Qwen y devuelve la activación media de la capa CAPA
    sobre los tokens de contenido (excluye los primeros 10 con normas anómalas).
    Retorna: (vector numpy [3584], n_tokens int)
    """
    almacen = {}

    def hook_fn(modulo, entrada, salida):
        almacen['act'] = salida.detach().float().cpu()

    handle = modelo.model.layers[CAPA].register_forward_hook(hook_fn)

    chat      = [{'role': 'user', 'content': texto}]
    input_ids = tokenizer.apply_chat_template(
        chat,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    )['input_ids'].to('cuda')

    n_tokens = input_ids.shape[1]

    with torch.no_grad():
        _ = modelo(input_ids=input_ids)

    handle.remove()

    tensor  = almacen['act'][0]           # [T, 3584]
    inicio  = min(10, n_tokens - 1)       # saltar tokens con normas anómalas
    act_vec = tensor[inicio:].mean(dim=0) # mean pooling → [3584]

    return act_vec.numpy(), n_tokens


# ── Cargar prompts ──────────────────────────────────────────────────
with open(PROMPTS_JSON, 'r', encoding='utf-8') as f:
    prompts = json.load(f)

grupos = {p['grupo'] for p in prompts}
print(f'✓ {len(prompts)} prompts cargados | grupos: {grupos}')
print(f'Total activaciones a extraer: {len(prompts) * 2} (ES + EN)\n')

# ── Loop principal ──────────────────────────────────────────────────
metadatos = []
errores   = []
total_ops = len(prompts) * 2

for i, entrada in enumerate(prompts):
    pid = entrada['id']

    for lang, campo in [('es', 'prompt_es'), ('en', 'prompt_en')]:
        op_num = i * 2 + (0 if lang == 'es' else 1) + 1
        texto  = entrada[campo]
        print(f'[{op_num:3d}/{total_ops}] {pid}_{lang} ...', end=' ', flush=True)

        try:
            act_vec, n_tok = extraer_activacion(texto)

            nombre_npy = f'{pid}_{lang}.npy'
            np.save(f'{DIR_SALIDA}/{nombre_npy}', act_vec)

            metadatos.append({
                'id'                 : pid,
                'lang'               : lang,
                'grupo'              : entrada['grupo'],
                'tema'               : entrada['tema'],
                'texto'              : texto,
                'senales_colombianas': entrada.get('senales_colombianas', []),
                'hipotesis_nla'      : entrada.get('hipotesis_nla', ''),
                'n_tokens'           : n_tok,
                'capa'               : CAPA,
                'd_model'            : 3584,
                'archivo_npy'        : nombre_npy,
            })
            print(f'✓  ({n_tok} tokens)')

        except Exception as e:
            errores.append({'id': pid, 'lang': lang, 'error': str(e)})
            print(f'✗  ERROR: {e}')

# ── Guardar metadatos ───────────────────────────────────────────────
ruta_meta = f'{DIR_SALIDA}/metadatos_activaciones.json'
with open(ruta_meta, 'w', encoding='utf-8') as f:
    json.dump(metadatos, f, ensure_ascii=False, indent=2)
    

print(f'\n{"="*55}')
print(f'✓ Activaciones guardadas : {len(metadatos)}')
print(f'✗ Errores                : {len(errores)}')
if errores:
    for e in errores:
        print(f'   {e}')
print(f'✓ Metadatos → {ruta_meta}')
print(f'\n🎉 Etapa 1 completada. Continúa con NLA_verbalizer.ipynb')